[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/15_mlp.ipynb)

# 🟡 Medium: SwiGLU MLP

*Core Ops & Layers*
Implement the **SwiGLU MLP** — the feed-forward block used in LLaMA, Mistral,
Gemma and most modern LLMs.

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x))
\odot \text{up\_proj}(x)\big)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$.

### Signature
```python
class SwiGLUMLP(nnx.Module):
    def __init__(self, d_model: int, d_ff: int, *, rngs: nnx.Rngs): ...
    def __call__(self, x): ...
```

### Requirements
- `self.gate_proj`: `nnx.Linear(d_model, d_ff)`
- `self.up_proj`:   `nnx.Linear(d_model, d_ff)`
- `self.down_proj`: `nnx.Linear(d_ff, d_model)`
- Activation: **SiLU** (a.k.a. Swish) — `jax.nn.silu`, or write `x * jax.nn.sigmoid(x)`

`nnx.Linear` is an allowed building block here — you are implementing the
*block*, not the linear layer. It also brings its own initialization
(lecun_normal kernel, zero bias), which is why you do not touch `nnx.Param`
directly in this problem.

### Why three projections and not two
A classic MLP is `down(act(up(x)))` — two matrices. SwiGLU splits the
expansion into two parallel projections and uses one to **gate** the other:
the network can suppress a channel by driving the gate negative, independently
of what the value projection says. That multiplicative interaction is what a
plain activation cannot express.

The cost is a third matrix. To keep the parameter count comparable, models
using SwiGLU shrink $d_{ff}$ from the classic $4d$ to about $\tfrac{8}{3}d$ —
LLaMA-7B uses $d_{ff} = 11008$ against $d_{model} = 4096$, a ratio of 2.69,
which is exactly $\tfrac{8}{3}$ rounded to a hardware-friendly multiple.

### The trap
It is easy to apply the activation to the wrong branch, or to both. Only the
**gate** goes through SiLU; `up_proj(x)` is multiplied in linearly. Swapping
them still runs, still trains, and is silently a different architecture — the
tests below pin down which branch is which.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class SwiGLUMLP(nnx.Module):
    """SwiGLU feed-forward block."""

    def __init__(self, d_model: int, d_ff: int, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, x):
        """(..., d_model) -> (..., d_model)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

mlp = SwiGLUMLP(d_model=8, d_ff=21, rngs=nnx.Rngs(params=0))
x = jax.random.normal(jax.random.key(1), (2, 5, 8))

print("in :", x.shape)
print("out:", mlp(x).shape, "(same as input)")
print("d_ff/d_model =", 21 / 8, "— LLaMA uses ~8/3, not 4")

params = nnx.state(mlp, nnx.Param)
print("param leaves:", len(jax.tree.leaves(params)), "(3 kernels + 3 biases)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("mlp")

# hint("mlp")      # stuck? nudge without the answer
# solution("mlp")  # spoiler: the reference implementation